# Deploying: Managed and Container Paths

**Phase 06** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-06/11-deploying-managed-and-container.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.

> **Note:** This lesson builds a service that must run as a long-lived process. The cells below show the full implementation but cannot run end-to-end in Colab. To run this lesson: clone the repo locally and use `uvicorn main:app` or `docker compose up`.


In [ ]:
!pip install -q anthropic fastapi httpx pydantic uvicorn

import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass  # Running locally — set ANTHROPIC_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
Lesson 11 - Deploying: Managed and Container Paths
Phase 06: Shipping

The complete Phase 06 FastAPI service, ready for deployment to Railway or Render.
Includes all required production endpoints:
  GET  /health    - health check (required by all platforms)
  POST /generate  - sync LLM generation
  GET  /stream    - SSE streaming generation

Deployment steps:
    Railway:  railway login && railway init && railway variables set ANTHROPIC_API_KEY=... && railway up
    Render:   render login && render deploy (requires render.yaml in project root)

Required environment variables:
    ANTHROPIC_API_KEY   - Your Anthropic API key (set via platform dashboard, never in code)
    PORT                - Set automatically by Railway/Render (default: 8000)
"""
import os

import anthropic
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse
from pydantic import BaseModel

### App setup

In [ ]:
app = FastAPI(
    title="AI Service",
    version="1.0.0",
    description="FastAPI service wrapping Claude. Deploy to Railway or Render.",
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # Restrict to your frontend domain in production
    allow_credentials=False,
    allow_methods=["GET", "POST"],
    allow_headers=["*"],
)

client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
MODEL = "claude-3-5-haiku-20241022"

### Data models

In [ ]:
class GenerateRequest(BaseModel):
    prompt: str
    max_tokens: int = 1024


class GenerateResponse(BaseModel):
    text: str
    model: str

### Endpoints

In [ ]:
@app.get("/health")
async def health() -> dict[str, str]:
    """
    Health check endpoint. Required by all deployment platforms.
    Railway, Render, Fly.io, Cloud Run all ping this to verify readiness.
    Must return HTTP 200. Should not call external services.
    """
    return {"status": "ok"}


@app.post("/generate", response_model=GenerateResponse)
async def generate(request: GenerateRequest) -> GenerateResponse:
    """Synchronous generation. Returns full response as JSON."""
    message = client.messages.create(
        model=MODEL,
        max_tokens=request.max_tokens,
        messages=[{"role": "user", "content": request.prompt}],
    )
    return GenerateResponse(text=message.content[0].text, model=MODEL)


@app.get("/stream")
async def stream(prompt: str, max_tokens: int = 1024) -> StreamingResponse:
    """
    Streaming generation via Server-Sent Events.
    Use EventSource in the browser or httpx.stream in a client script.
    Prompt passed as query parameter (EventSource is GET-only).
    """

    def generate_stream():
        with client.messages.stream(
            model=MODEL,
            max_tokens=max_tokens,
            messages=[{"role": "user", "content": prompt}],
        ) as stream_ctx:
            for text in stream_ctx.text_stream:
                # SSE format requires double newline after each data line
                safe_text = text.replace("\n", " ")
                yield f"data: {safe_text}\n\n"
        yield "data: [DONE]\n\n"

    return StreamingResponse(
        generate_stream(),
        media_type="text/event-stream",
        headers={
            "Cache-Control": "no-cache",
            "X-Accel-Buffering": "no",  # Prevent nginx buffering
        },
    )


# ---------------------------------------------------------------------------
# Deployment verification script
# ---------------------------------------------------------------------------

### Demo

In [ ]:
"""
Run this script after deploying to verify your service is live.
Usage: BASE_URL=https://your-service.railway.app python main.py
"""
import sys

import httpx

base_url = os.environ.get("BASE_URL", "http://localhost:8000").rstrip("/")
print(f"Verifying deployment at: {base_url}")
errors = []

# Test 1: Health check
try:
    resp = httpx.get(f"{base_url}/health", timeout=10)
    if resp.status_code == 200 and resp.json().get("status") == "ok":
        print(f"  [PASS] GET /health -> {resp.status_code}")
    else:
        errors.append(f"  [FAIL] GET /health -> {resp.status_code}: {resp.text}")
except Exception as exc:
    errors.append(f"  [FAIL] GET /health -> {exc}")

# Test 2: Generate
try:
    resp = httpx.post(
        f"{base_url}/generate",
        json={"prompt": "Say 'deployment verified' and nothing else."},
        timeout=30,
    )
    if resp.status_code == 200 and "text" in resp.json():
        print(f"  [PASS] POST /generate -> {resp.status_code}")
        print(f"         Response: {resp.json()['text'][:60]}")
    else:
        errors.append(f"  [FAIL] POST /generate -> {resp.status_code}: {resp.text}")
except Exception as exc:
    errors.append(f"  [FAIL] POST /generate -> {exc}")

# Test 3: Streaming
try:
    tokens_received = 0
    done_received = False
    with httpx.stream(
        "GET",
        f"{base_url}/stream",
        params={"prompt": "Count: 1, 2, 3"},
        timeout=30,
    ) as r:
        for line in r.iter_lines():
            if line.startswith("data: "):
                token = line[6:]
                if token == "[DONE]":
                    done_received = True
                    break
                tokens_received += 1
    if tokens_received > 0 and done_received:
        print(f"  [PASS] GET /stream -> {tokens_received} tokens, [DONE] received")
    else:
        errors.append(
            f"  [FAIL] GET /stream -> tokens={tokens_received}, done={done_received}"
        )
except Exception as exc:
    errors.append(f"  [FAIL] GET /stream -> {exc}")

if errors:
    print("\nFailed checks:")
    for err in errors:
        print(err)
    sys.exit(1)
else:
    print(f"\nAll checks passed. Service is live at {base_url}")

---

## Self-Check

Answer these without running code first:

**1. Which deployment path gives you the fastest working URL with the lowest operational burden?**

- A. AWS ECS/Fargate with an ALB and auto-scaling group.
- B. Railway or Render: push your Dockerfile, get an HTTPS URL in 15 minutes.
- C. GCP Kubernetes Engine for maximum scalability.
- D. A raw EC2 instance where you install Docker and run the container yourself.

**2. What is the root cause and how do you fix it?**

- A. Railway does not support FastAPI. Switch to Flask.
- B. Your FastAPI app does not have a GET /health endpoint. Add one that returns HTTP 200.
- C. The Dockerfile EXPOSE directive is missing. Add EXPOSE 8000.
- D. Railway requires a render.yaml file. Add it to your project root.

**3. What is wrong with this approach?**

- A. ENV only works in docker-compose.yml, not Dockerfile.
- B. The API key is baked into the Docker image, which may be pushed to a registry and exposed.
- C. The ENV instruction requires the variable to be exported separately.
- D. Nothing. Baking secrets into images is standard practice for production deployments.

**4. What does this requirement mean for your deployment?**

- A. Nothing. Railway encrypts all traffic so it is compliant.
- B. You must migrate to AWS ECS or another service that runs inside your own VPC.
- C. Add a VPN tunnel between Railway and your AWS VPC.
- D. Use Railway's enterprise tier, which provides VPC isolation.

**5. What is causing this and how do you fix it?**

- A. The Anthropic API has a cold start problem. Use a different model.
- B. Render (and Railway) scale to zero when idle. The first request wakes the container. Disable scale-to-zero or use a cron ping to keep it warm.
- C. Your FastAPI app takes 8 seconds to load. Optimize your import statements.
- D. The health check is running during the first request. Move it to a separate thread.

**6. How do you access the error logs on Railway?**

- A. SSH into the Railway container and check /var/log/app.log.
- B. Run 'railway logs' in your terminal, or use the Logs tab in the Railway dashboard.
- C. Set LOG_LEVEL=DEBUG in your Dockerfile and redeploy.
- D. Add a /logs endpoint to your FastAPI app that returns the last 100 log lines.

**7. Which platform is correct and why?**

- A. AWS ECS, because all serious companies use AWS.
- B. Railway, because it matches the constraints: low volume, no compliance, fast deadline, no AWS expertise needed.
- C. AWS ECS, because you will need it when traffic grows.
- D. Neither. Build the service on a raw VM to avoid platform lock-in.

**8. What is wrong and how do you fix it?**

- A. Railway does not support uvicorn. Use gunicorn instead.
- B. The hard-coded port 8000 in CMD does not match the PORT env var Railway sets. Use CMD ['sh', '-c', 'uvicorn main:app --host 0.0.0.0 --port $PORT'] to read the port at runtime.
- C. The --host 0.0.0.0 flag is insecure. Railway requires --host 127.0.0.1.
- D. Add EXPOSE 4732 to your Dockerfile to match the assigned port.

_Answers are in `checks.json` in the lesson directory._